# 01 Collect Yearly Data

Canonical data collection notebook for the refactored AMR workflow.
This notebook builds yearly pair counts, yearly background marginals, runs the unified LLM screen, and exports `yearly_amr_screened_pairs.parquet`.


In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
from pathlib import Path
import pandas as pd
from amr.filtering import (
    UNIFIED_AMR_INTEREST_SYSTEM_PROMPT,
    build_user_prompt,
    build_yearly_screened_pairs,
    prepare_yearly_candidate_pmids,
    run_unified_llm_filter,
)
from amr.llm import OpenAIConfig, OpenAILanguageModel
from amr.pipelines.yearly_collection import (
    build_collection_preflight_summary,
    collect_yearly_amr_dataset,
    collect_yearly_background_counts,
)
from amr.pubmed.queries import (
    build_antibiotic_pathogen_query,
    build_pathogen_tiab_query,
    prepare_antibiotic_pubmed_query_tiab,
)


## Glossary

- **AMR**: Antimicrobial Resistance.
- **SI**: Scientific Interest score, defined as residual AMR attention after background adjustment.
- **TIAB**: Title/Abstract PubMed field selector (search terms constrained to title or abstract text).
- **PMID**: PubMed Identifier for a publication record.
- **LLM**: Large Language Model used for abstract screening.
- **BG / background counts**: Yearly publication marginals for pathogens or antibiotics used in SI baseline modeling.
- **Slice**: A chosen year interval (`SLICE_START_YEAR` to `SLICE_END_YEAR`) processed in one run.


## Configuration
Set paths and run options. Use a real PubMed contact email and valid LLM credentials.


In [ ]:
PATHOGENS_PATH = Path("bacteria_synonym_lookup.csv")
ANTIBIOTIC_METADATA_PATH = Path("antibiotic_list2.xlsx")
OUTPUT_YEARLY_AMR_PATH = Path("yearly_amr_pairs.parquet")
OUTPUT_YEARLY_SCREEN_PATH = Path("yearly_amr_screened_pairs.parquet")
OUTPUT_UNIFIED_DECISIONS_PATH = Path("yearly_amr_unified_decisions.parquet")
OUTPUT_PATHOGEN_BG_PATH = Path("pathogen_tiab_counts.parquet")
OUTPUT_ANTIBIOTIC_BG_PATH = Path("antibiotic_tiab_counts.parquet")
INTERVAL_PMIDS_PATH = Path("pmids_results.parquet")
YEARLY_PAIR_PMID_PATH = Path("yearly_pair_pmid.parquet")
PUBMED_METADATA_CACHE_PATH = Path("pubmed_pmid_metadata.parquet")
ABSTRACT_CACHE_PATH = Path("pmid_abstracts.parquet")
PUBMED_EMAIL = "xfcosta@gmail.com"  # replace with a real contact email

# Auto-resume behavior: all collection/filtering stages reuse on-disk outputs
# and continue only the missing work.

# Partition-friendly execution window: run a slice now, merge into canonical stores.
TOTAL_START_YEAR = 1970
TOTAL_END_YEAR = 2026
TOTAL_MIN_DATE = f"{TOTAL_START_YEAR}/01/01"
TOTAL_MAX_DATE = f"{TOTAL_END_YEAR}/12/31"
SLICE_START_YEAR = 2020
SLICE_END_YEAR = 2026
MIN_DATE = f"{SLICE_START_YEAR}/01/01"
MAX_DATE = f"{SLICE_END_YEAR}/12/31"
INTERVAL_YEAR_GAP = 5
# Existing notebook intent: AMR-focused retrieval before unified LLM relevance screen.
EXTRA_AMR_KEYWORDS = [
    "\"antimicrobial resistance\"[Title/Abstract] OR \"antibiotic resistance\"[Title/Abstract]"
]
RUN_LLM_FILTER = True
MAX_LLM_CALLS = None
# Editable per-run prompt (single source of truth for this notebook execution)
UNIFIED_SYSTEM_PROMPT = UNIFIED_AMR_INTEREST_SYSTEM_PROMPT
# Throughput/reliability controls for unified LLM filter
LLM_MAX_WORKERS = 2
LLM_BATCH_SIZE = 8
FAILED_RETRY_PATH = Path("yearly_amr_unified_failures.parquet")
MAX_FAILURE_RETRIES = 5
ABSTRACT_FETCH_BATCH_SIZE = 200
PREFETCH_ABSTRACTS = True
# Batch-style collection logging controls
COLLECTION_VERBOSE = 1  # 0=silent, 1=batch summaries, 2=detailed per interval
COLLECTION_PROGRESS_EVERY = 300
BACKGROUND_FETCH_THRESHOLD = 5000
BACKGROUND_METADATA_BATCH_SIZE = 200
# Unified filter logging cadence
UNIFIED_PROGRESS_EVERY = 200


In [4]:
from textwrap import fill

print("PRETTY PRINT (80 COLS) OF UNIFIED_SYSTEM_PROMPT:\n")
print("\n".join(fill(line, width=80) if line.strip() else "" for line in UNIFIED_SYSTEM_PROMPT.splitlines()))


PRETTY PRINT (80 COLS) OF UNIFIED_SYSTEM_PROMPT:

You are screening PubMed abstracts for pathogen-antibiotic antimicrobial
resistance evidence.

A study should be included only when BOTH are true:
1) The abstract explicitly reports that the specified pathogen shows resistance
/ reduced susceptibility / treatment failure due to resistance for the specified
antibiotic.
2) The study is scientifically relevant for real-world AMR interest (clinical,
surveillance, epidemiological, environmental, food, veterinary, or naturally
occurring strain contexts), and not primarily a laboratory model-system
experiment where AMR is incidental.

Decision labels:
- yes: both conditions are clearly satisfied.
- no: at least one condition is clearly not satisfied.
- uncertain: abstract missing, ambiguous, or insufficient to judge confidently.

Return compact JSON with keys: decision (yes/no/uncertain) and rationale (one
short sentence).


## Load Pathogen and Antibiotic Lists


In [5]:
bacteria_synonym_df = pd.read_csv(PATHOGENS_PATH)
antibiotic_df = pd.read_excel(ANTIBIOTIC_METADATA_PATH)

pathogens = sorted(bacteria_synonym_df["bacteria name"].dropna().astype(str).str.strip().unique())
antibiotics = sorted(antibiotic_df["Antibiotic"].dropna().astype(str).str.strip().unique())

print(f"Pathogens: {len(pathogens):,}")
print(f"Antibiotics: {len(antibiotics):,}")


Pathogens: 19,956
Antibiotics: 245


## Collect Yearly Pair Dataset and Background Marginals


## PubMed Query Logic for Abstract Collection

The collection query is assembled per `(pathogen, antibiotic)` pair with four components:

1. **Antibiotic clause**: antibiotic name in `Abstract`, including split variants for hyphen/slash names (for example `co-trimoxazole` as exact phrase OR split terms).
2. **Pathogen clause**: pathogen + known synonyms in `Abstract` (`OR`-joined).
3. **AMR clause**: additional AMR-focused condition from `EXTRA_AMR_KEYWORDS` in `Title/Abstract`.
4. **Date clause**: publication date restricted to `[MIN_DATE, MAX_DATE]`.

Final query shape:

`(antibiotic_clause AND pathogen_clause) AND extra_amr_clause AND date_range_clause`


In [6]:
sample_pathogen = pathogens[0]
sample_antibiotic = antibiotics[0]
sample_query = build_antibiotic_pathogen_query(
    sample_antibiotic,
    sample_pathogen,
    bacteria_synonym_df,
    extra_clauses=EXTRA_AMR_KEYWORDS,
    min_date=MIN_DATE,
    max_date=MAX_DATE,
)
print("Sample pathogen:", sample_pathogen)
print("Sample antibiotic:", sample_antibiotic)
print("\nSample PubMed query:\n")
print(sample_query)


Sample pathogen: abditibacterium utsteinense
Sample antibiotic: Amikacin

Sample PubMed query:

("Amikacin"[Abstract] AND ("abditibacterium utsteinense"[Abstract])) AND "antimicrobial resistance"[Title/Abstract] OR "antibiotic resistance"[Title/Abstract] AND ("2020/01/01"[Date - Publication] : "2026/12/31"[Date - Publication])


In [7]:
print("EXAMPLE PUBMED QUERIES (MULTIPLE VIEWS)\n")

example_pairs = list(zip(pathogens[:3], antibiotics[:3]))
for idx, (p_name, a_name) in enumerate(example_pairs, start=1):
    pathogen_clause = build_pathogen_tiab_query(p_name, bacteria_synonym_df)
    antibiotic_clause = prepare_antibiotic_pubmed_query_tiab(a_name)
    pair_query = build_antibiotic_pathogen_query(
        a_name,
        p_name,
        bacteria_synonym_df,
        extra_clauses=EXTRA_AMR_KEYWORDS,
        min_date=MIN_DATE,
        max_date=MAX_DATE,
    )
    print(f"Example {idx}: {p_name} vs {a_name}")
    print("- Pathogen TIAB clause:")
    print(pathogen_clause)
    print("- Antibiotic TIAB clause:")
    print(antibiotic_clause)
    print("- Final pair + AMR + date query:")
    print(pair_query)
    print("\n" + "-" * 100 + "\n")


EXAMPLE PUBMED QUERIES (MULTIPLE VIEWS)

Example 1: abditibacterium utsteinense vs Amikacin
- Pathogen TIAB clause:
("abditibacterium utsteinense"[Title/Abstract])
- Antibiotic TIAB clause:
"Amikacin"[Title/Abstract]
- Final pair + AMR + date query:
("Amikacin"[Abstract] AND ("abditibacterium utsteinense"[Abstract])) AND "antimicrobial resistance"[Title/Abstract] OR "antibiotic resistance"[Title/Abstract] AND ("2020/01/01"[Date - Publication] : "2026/12/31"[Date - Publication])

----------------------------------------------------------------------------------------------------

Example 2: abiotrophia defectiva vs Amoxicillin
- Pathogen TIAB clause:
("abiotrophia defectiva"[Title/Abstract] OR "streptococcus defectivus"[Title/Abstract])
- Antibiotic TIAB clause:
"Amoxicillin"[Title/Abstract]
- Final pair + AMR + date query:
("Amoxicillin"[Abstract] AND ("abiotrophia defectiva"[Abstract] OR "streptococcus defectivus"[Abstract])) AND "antimicrobial resistance"[Title/Abstract] OR "antibiot

In the next code cell, the notebook will automatically gather research paper IDs from PubMed for each pathogen-antibiotic combination in your selected year range, convert those papers into yearly counts, and then save the results to local files so future runs can continue from where you left off. It also collects yearly background publication totals for each pathogen and antibiotic, which are later used to calculate scientific interest in a fair, comparable way.


## Resume Preflight

Estimate how much work is already cached on disk before the resumable collection step starts.


In [8]:
preflight = build_collection_preflight_summary(
    pathogens=pathogens,
    antibiotics=antibiotics,
    min_date=MIN_DATE,
    max_date=MAX_DATE,
    interval_year_gap=INTERVAL_YEAR_GAP,
    intervals_output_path=INTERVAL_PMIDS_PATH,
    yearly_pair_pmid_path=YEARLY_PAIR_PMID_PATH,
    yearly_output_path=OUTPUT_YEARLY_AMR_PATH,
    pathogen_bg_path=OUTPUT_PATHOGEN_BG_PATH,
    antibiotic_bg_path=OUTPUT_ANTIBIOTIC_BG_PATH,
    background_start_year=SLICE_START_YEAR,
    background_end_year=SLICE_END_YEAR,
    total_min_date=TOTAL_MIN_DATE,
    total_max_date=TOTAL_MAX_DATE,
    total_background_start_year=TOTAL_START_YEAR,
    total_background_end_year=TOTAL_END_YEAR,
)

def _print_window_report(label: str, window_summary: dict) -> None:
    interval_cache = window_summary["interval_cache"]
    pathogen_bg_cache = window_summary["pathogen_bg_cache"]
    antibiotic_bg_cache = window_summary["antibiotic_bg_cache"]
    print(label)
    print(f"Date window: {window_summary['min_date']} .. {window_summary['max_date']}")
    print(f"Intervals in window: {len(window_summary['intervals'])} -> {window_summary['intervals']}")
    print(
        f"Interval cache coverage: {interval_cache['cached_rows']:,} / {interval_cache['expected_rows']:,} "
        f"rows ({interval_cache['cached_pct']:.2f}%)"
    )
    print(f"Missing interval rows still to collect: {interval_cache['missing_rows']:,}")
    print(
        f"Cached interval breadth: {interval_cache['unique_pathogens']:,} pathogens, "
        f"{interval_cache['unique_antibiotics']:,} antibiotics, "
        f"{interval_cache['unique_pairs']:,} unique pairs"
    )
    print(f"Existing yearly AMR rows in window: {window_summary['yearly_rows']:,}")
    print(f"Existing yearly pair PMID rows in window: {window_summary['yearly_pair_pmid_rows']:,}")
    print(
        f"Pathogen background cache: {pathogen_bg_cache['cached_rows']:,} / "
        f"{pathogen_bg_cache['expected_rows']:,} rows ({pathogen_bg_cache['cached_pct']:.2f}%)"
    )
    pathogen_missing_pct = (100.0 - pathogen_bg_cache['cached_pct']) if pathogen_bg_cache['expected_rows'] else 0.0
    print(
        f"Missing pathogen background rows: {pathogen_bg_cache['missing_rows']:,} "
        f"({pathogen_missing_pct:.2f}%)"
    )
    if pathogen_bg_cache['warning']:
        print(f"WARNING: {pathogen_bg_cache['warning']}")
    print(
        f"Antibiotic background cache: {antibiotic_bg_cache['cached_rows']:,} / "
        f"{antibiotic_bg_cache['expected_rows']:,} rows ({antibiotic_bg_cache['cached_pct']:.2f}%)"
    )
    antibiotic_missing_pct = (100.0 - antibiotic_bg_cache['cached_pct']) if antibiotic_bg_cache['expected_rows'] else 0.0
    print(
        f"Missing antibiotic background rows: {antibiotic_bg_cache['missing_rows']:,} "
        f"({antibiotic_missing_pct:.2f}%)"
    )
    if antibiotic_bg_cache['warning']:
        print(f"WARNING: {antibiotic_bg_cache['warning']}")
    print()

print("COLLECTION PREFLIGHT\n")
print(f"Existing yearly AMR span on disk: {preflight['yearly_year_min']} .. {preflight['yearly_year_max']}")
print(f"Existing yearly AMR rows on disk: {preflight['yearly_rows']:,}")
print(f"Existing yearly pair PMID rows on disk: {preflight['yearly_pair_pmid_rows']:,}\n")
print("TOTAL WINDOW SUMMARY")
_print_window_report("", preflight['total_window'])
print("CURRENT SLICE SUMMARY")
_print_window_report("", preflight['run_window'])

if preflight['run_window']['interval_cache']['missing_rows'] > 0:
    print("WARNING: collection is resumable, but a non-trivial amount of interval PubMed work remains for the current slice.")
    if preflight['run_window']['interval_cache']['cached_pct'] < 50:
        print("Current-slice interval cache coverage is below 50%, so this run may still be expensive.")


COLLECTION PREFLIGHT

Existing yearly AMR span on disk: 1975 .. 2026
Existing yearly AMR rows on disk: 41,284
Existing yearly pair PMID rows on disk: 0

TOTAL WINDOW SUMMARY

Date window: 1970/01/01 .. 2026/12/31
Intervals in window: 12 -> [('1970/01/01', '1974/12/31'), ('1975/01/01', '1979/12/31'), ('1980/01/01', '1984/12/31'), ('1985/01/01', '1989/12/31'), ('1990/01/01', '1994/12/31'), ('1995/01/01', '1999/12/31'), ('2000/01/01', '2004/12/31'), ('2005/01/01', '2009/12/31'), ('2010/01/01', '2014/12/31'), ('2015/01/01', '2019/12/31'), ('2020/01/01', '2024/12/31'), ('2025/01/01', '2026/12/31')]
Interval cache coverage: 1,901,000 / 58,670,640 rows (3.24%)
Missing interval rows still to collect: 56,769,640
Cached interval breadth: 788 pathogens, 248 antibiotics, 192,960 unique pairs
Existing yearly AMR rows in window: 41,284
Existing yearly pair PMID rows in window: 0
Pathogen background cache: 44,572 / 1,137,492 rows (3.92%)
Missing pathogen background rows: 1,092,920 (96.08%)
Antibiotic

In [ ]:
print("Auto-resume: collection stages will reuse saved outputs and continue missing work only.")

yearly_amr_df, yearly_pair_pmid_df, enriched_pair_pmid_df = collect_yearly_amr_dataset(
    pathogens=pathogens,
    antibiotics=antibiotics,
    bacteria_synonym_df=bacteria_synonym_df,
    min_date=MIN_DATE,
    max_date=MAX_DATE,
    email=PUBMED_EMAIL,
    interval_year_gap=INTERVAL_YEAR_GAP,
    intervals_output_path=INTERVAL_PMIDS_PATH,
    yearly_pair_pmid_output_path=YEARLY_PAIR_PMID_PATH,
    metadata_cache_path=PUBMED_METADATA_CACHE_PATH,
    output_path=OUTPUT_YEARLY_AMR_PATH,
    extra_keywords=EXTRA_AMR_KEYWORDS,
    collection_verbose=COLLECTION_VERBOSE,
    collection_progress_every=COLLECTION_PROGRESS_EVERY,
)


Auto-resume: collection stages will reuse saved outputs and continue missing work only.


In [ ]:

pathogen_bg_df, antibiotic_bg_df = collect_yearly_background_counts(
    pathogens=pathogens,
    antibiotics=antibiotics,
    bacteria_synonym_df=bacteria_synonym_df,
    start_year=SLICE_START_YEAR,
    end_year=SLICE_END_YEAR,
    email=PUBMED_EMAIL,
    pathogen_output_path=OUTPUT_PATHOGEN_BG_PATH,
    antibiotic_output_path=OUTPUT_ANTIBIOTIC_BG_PATH,
    fetch_threshold=BACKGROUND_FETCH_THRESHOLD,
    metadata_batch_size=BACKGROUND_METADATA_BATCH_SIZE,
)

print(f"yearly AMR rows: {len(yearly_amr_df):,}")
print(f"yearly pair PMID rows: {len(yearly_pair_pmid_df):,}")
print(f"newly enriched pair PMID rows this run: {len(enriched_pair_pmid_df):,}")
print(f"pathogen bg rows: {len(pathogen_bg_df):,}")
print(f"antibiotic bg rows: {len(antibiotic_bg_df):,}")


## Unified LLM Filtering
Single strict pass: AMR evidence + scientific-interest relevance.


## Unified LLM Prompt Templates
This workflow uses a single strict LLM screen:
- **System prompt**: policy for AMR evidence + scientific-interest relevance.
- **User prompt template**: PMID-specific payload (pathogen, antibiotic, title, abstract).
**Prompt governance note:** if you change this prompt between slices, document it so downstream results remain interpretable.


In [ ]:
print("UNIFIED SYSTEM PROMPT:\n")
print(UNIFIED_SYSTEM_PROMPT)
print("\n" + "=" * 80 + "\n")
print("USER PROMPT TEMPLATE EXAMPLE:\n")
print(
    build_user_prompt(
        pmid="12345678",
        pathogen="Escherichia coli",
        antibiotic="Ciprofloxacin",
        title="Template title",
        abstract="Template abstract text.",
    )
)


In [ ]:
candidate_df = prepare_yearly_candidate_pmids(yearly_amr_df)
candidate_df = candidate_df.loc[
    candidate_df["year"].between(SLICE_START_YEAR, SLICE_END_YEAR)
].reset_index(drop=True)
print(f"Yearly candidate PMIDs in slice [{SLICE_START_YEAR}, {SLICE_END_YEAR}]: {len(candidate_df):,}")

resume_df = None
if OUTPUT_UNIFIED_DECISIONS_PATH.exists():
    resume_df = pd.read_parquet(OUTPUT_UNIFIED_DECISIONS_PATH)
    print(f"Found existing unified decisions: {len(resume_df):,} rows. Missing candidates will be resumed automatically.")

if RUN_LLM_FILTER:
    llm = OpenAILanguageModel(OpenAIConfig(model="gpt-4o-mini", max_tokens=256, temperature=0))
    unified_df = run_unified_llm_filter(
        candidate_df,
        llm,
        email=PUBMED_EMAIL,
        system_prompt=UNIFIED_SYSTEM_PROMPT,
        output_path=str(OUTPUT_UNIFIED_DECISIONS_PATH),
        resume_from=resume_df,
        auto_resume_from_output=True,
        llm_max_workers=LLM_MAX_WORKERS,
        llm_batch_size=LLM_BATCH_SIZE,
        abstract_fetch_batch_size=ABSTRACT_FETCH_BATCH_SIZE,
        prefetch_abstracts=PREFETCH_ABSTRACTS,
        failed_retry_path=str(FAILED_RETRY_PATH),
        max_failure_retries=MAX_FAILURE_RETRIES,
        abstract_cache_path=str(ABSTRACT_CACHE_PATH),
        max_llm_calls=MAX_LLM_CALLS,
        progress_every=UNIFIED_PROGRESS_EVERY,
        verbose=1,
    )
else:
    if resume_df is None or resume_df.empty:
        raise FileNotFoundError(
            f"No unified decisions file found at {OUTPUT_UNIFIED_DECISIONS_PATH}. "
            "Enable RUN_LLM_FILTER to create it."
        )
    unified_df = resume_df
    print(f"Loaded unified decisions from {OUTPUT_UNIFIED_DECISIONS_PATH} ({len(unified_df):,} rows).")

print(unified_df["unified_decision"].value_counts(dropna=False))
ABSTRACT_FETCH_BATCH_SIZE = 200
PREFETCH_ABSTRACTS = True


## Export Canonical Screened Yearly Dataset
Includes decisions `yes` + `uncertain` by default.


In [ ]:
yearly_screened_df = build_yearly_screened_pairs(
    yearly_amr_df,
    unified_df,
    include_decisions=("yes", "uncertain"),
    output_path=str(OUTPUT_YEARLY_SCREEN_PATH),
    merge_with_existing_output=True,
)

print(f"Screened yearly rows: {len(yearly_screened_df):,}")
print(f"Unique pairs: {yearly_screened_df[['pathogen','antibiotic']].drop_duplicates().shape[0]:,}")
print(f"Year span: {int(yearly_screened_df['year'].min())} - {int(yearly_screened_df['year'].max())}")
yearly_screened_df.head(10)

